In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast # Used for safely evaluating string literals containing Python structures

# Load the datasets
try:
    movies_df = pd.read_csv('tmdb_5000_movies.csv')
    credits_df = pd.read_csv('tmdb_5000_credits.csv')
    print("Datasets loaded successfully!")
except FileNotFoundError:
    print("Error: Make sure 'tmdb_5000_movies.csv' and 'tmdb_5000_credits.csv' are in the same directory.")
    print("You can download them from: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata")
    exit() # Exit if files are not found

print("\nMovies DataFrame Head:")
print(movies_df.head())
print("\nCredits DataFrame Head:")
print(credits_df.head())

Datasets loaded successfully!

Movies DataFrame Head:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id":

In [3]:
# Merge the two dataframes
# The 'title' column exists in both, but 'movie_id' is a better common key for credits
# Let's rename 'movie_id' in credits_df to 'id' to match movies_df for merging
credits_df.columns = ['id', 'title_x', 'cast', 'crew']
df = movies_df.merge(credits_df, on='id')

print("\nMerged DataFrame Head:")
print(df.head())
print("\nMerged DataFrame Columns:")
print(df.columns)


Merged DataFrame Head:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en

In [4]:
# Select relevant features
features = ['genres', 'keywords', 'cast', 'crew', 'overview', 'title']
df = df[features]

# Handle missing values, especially for 'overview'
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True) # Reset index after dropping rows

print("\nDataFrame after feature selection and dropping NaNs:")
print(df.head())
print(f"Remaining rows: {len(df)}")


# Helper function to parse JSON-like strings
def parse_json_features(obj):
    if isinstance(obj, str):
        try:
            items = ast.literal_eval(obj)
            if isinstance(items, list):
                return [i['name'] for i in items if 'name' in i]
        except (ValueError, SyntaxError):
            pass # Return empty list if parsing fails
    return []

# Apply the parsing function to 'genres' and 'keywords'
df['genres'] = df['genres'].apply(parse_json_features)
df['keywords'] = df['keywords'].apply(parse_json_features)

# For 'cast', we'll take the top 3 actors
def get_top_actors(obj, limit=3):
    if isinstance(obj, str):
        try:
            actors = ast.literal_eval(obj)
            if isinstance(actors, list):
                return [i['name'] for i in actors if 'name' in i][:limit]
        except (ValueError, SyntaxError):
            pass
    return []

df['cast'] = df['cast'].apply(get_top_actors)

# For 'crew', we only need the director
def get_director(obj):
    if isinstance(obj, str):
        try:
            crew_list = ast.literal_eval(obj)
            if isinstance(crew_list, list):
                for member in crew_list:
                    if member['job'] == 'Director':
                        return member['name']
        except (ValueError, SyntaxError):
            pass
    return np.nan # Return NaN if director not found or parsing fails

df['director'] = df['crew'].apply(get_director)

# Drop the original 'crew' column as we now have 'director'
df.drop('crew', axis=1, inplace=True)

# Drop rows where 'director' is NaN (movies without a director)
df.dropna(subset=['director'], inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nDataFrame after parsing JSON features and extracting director:")
print(df.head())
print(f"Remaining rows: {len(df)}")


DataFrame after feature selection and dropping NaNs:
                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...   
2  [{"id": 470, "name": "spy"}, {"id": 818, "name...   
3  [{"id": 849, "name": "dc comics"}, {"id": 853,...   
4  [{"id": 818, "name": "based on novel"}, {"id":...   

                                                cast  \
0  [{"cast_id": 242, "character": "Jake Sully", "...   
1  [{"cast_id": 4, "character": "Captain Jack Spa...   
2  [{"cast_id": 1, "character": "James Bond", "cr...   
3  [{"cast_id": 2, "character": "Bruce Wayne / B

In [5]:
# Clean up features: convert to lowercase and remove spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return '' # Return empty string for non-string, non-list types

df['genres'] = df['genres'].apply(clean_data)
df['keywords'] = df['keywords'].apply(clean_data)
df['cast'] = df['cast'].apply(clean_data)
df['director'] = df['director'].apply(clean_data)

# 'overview' is already a single string, just convert to lowercase
df['overview'] = df['overview'].apply(lambda x: x.lower() if isinstance(x, str) else '')


# Create a 'soup' column by combining all cleaned features
def create_soup(row):
    return ' '.join(row['keywords']) + ' ' + ' '.join(row['cast']) + ' ' + \
           row['director'] + ' ' + ' '.join(row['genres']) + ' ' + row['overview']

df['soup'] = df.apply(create_soup, axis=1)

print("\nDataFrame with cleaned features and 'soup' column:")
print(df[['title', 'soup']].head())


DataFrame with cleaned features and 'soup' column:
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                soup  
0  cultureclash future spacewar spacecolony socie...  
1  ocean drugabuse exoticisland eastindiatradingc...  
2  spy basedonnovel secretagent sequel mi6 britis...  
3  dccomics crimefighter terrorist secretidentity...  
4  basedonnovel mars medallion spacetravel prince...  


In [6]:
# Initialize TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the 'soup' column
tfidf_matrix = tfidf.fit_transform(df['soup'])

print("\nTF-IDF Matrix Shape:", tfidf_matrix.shape)
# The shape will be (number_of_movies, number_of_unique_words_after_vectorization)


TF-IDF Matrix Shape: (4770, 35215)


In [7]:
# Calculate the cosine similarity matrix
# linear_kernel is faster than cosine_similarity for normalized vectors
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("\nCosine Similarity Matrix Shape:", cosine_sim.shape)
print("Example similarity score (first movie with second movie):", cosine_sim[0, 1])


Cosine Similarity Matrix Shape: (4770, 4770)
Example similarity score (first movie with second movie): 0.01462662323360778


In [9]:
# Create a reverse mapping of movie titles to DataFrame indices
title_to_index = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend_movies(title, cosine_sim_matrix=cosine_sim, df_data=df, top_n=10):
    """
    Recommends top_n similar movies based on the given movie title.

    Args:
        title (str): The title of the movie for which to find recommendations.
        cosine_sim_matrix (np.array): The pre-computed cosine similarity matrix.
        df_data (pd.DataFrame): The DataFrame containing movie titles and other data.
        top_n (int): The number of recommendations to return.

    Returns:
        pd.Series: A Series of recommended movie titles, or a message if the movie is not found.
    """
    if title not in title_to_index:
        return f"Sorry, the movie '{title}' is not found in our database. Please check the spelling."

    # Get the index of the movie that matches the title
    idx = title_to_index[title]

    # Get the pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n+1 most similar movies (excluding the movie itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top_n most similar movie titles
    return df_data['title'].iloc[movie_indices]

In [ ]:
# Example Usage:
print("--- Movie Recommendation System ---")
while True:
    movie_name = input("\nEnter a movie title (or type 'exit' to quit): ").strip()
    if movie_name.lower() == 'exit':
        print("Exiting application. Goodbye!")
        break

    recommendations = recommend_movies(movie_name)

    if isinstance(recommendations, str): # Error message
        print(recommendations)
    else:
        print(f"\nRecommendations for '{movie_name}':")
        if not recommendations.empty:
            for i, rec_movie in enumerate(recommendations, 1):
                print(f"{i}. {rec_movie}")
        else:
            print("No recommendations found for this movie. It might be too unique!")

    print("-" * 40)

--- Movie Recommendation System ---



Enter a movie title (or type 'exit' to quit):  iron man


Sorry, the movie 'iron man' is not found in our database. Please check the spelling.
----------------------------------------



Enter a movie title (or type 'exit' to quit):  The Dark Knight Rises



Recommendations for 'The Dark Knight Rises':
1. The Dark Knight
2. Batman Returns
3. Batman Begins
4. Batman Forever
5. Batman
6. Batman: The Dark Knight Returns, Part 2
7. Batman & Robin
8. Batman v Superman: Dawn of Justice
9. Slow Burn
10. Defendor
----------------------------------------
